## Boosting

In [ ]:
Boosting (изначально называвшийся hypothesis boosting) — это общее название 
для любых ансамблевых методов, которые позволяют объединить несколько слабых
моделей в одну сильную. Основная идея большинства методов бустинга — обучать
предсказатели последовательно, причём каждый пытается исправить ошибки предыдущего. 
Существует много методов бустинга, но самые популярные — AdaBoost 
(сокращение от adaptive boosting) и градиентный бустинг. Начнём с AdaBoost.

## AdaBoost

**Идея:** модели обучаются ПОСЛЕДОВАТЕЛЬНО. Каждая новая исправляет ошибки
предыдущей — не улучшает её, а ДОБАВЛЯЕТСЯ рядом с ней.

![Figure 6-7 — последовательное обучение AdaBoost](6-7.png)

*Внизу: данные с весами (размер кружка = вес точки). В середине: обучение модели.
Сверху: граница решений этой модели. Стрелки = данные с новыми весами уходят дальше.*

### Механика — через вес точек
У каждой точки есть вес: «насколько громко она кричит».
1. Сначала вес у всех одинаковый.
2. Учим базовую модель (например дерево), смотрим, где она ошиблась.
3. Ошибочные точки получают БОЛЬШИЙ вес.
4. Учим вторую модель — она вынуждена обращать внимание на «тяжёлые» точки.
5. Снова ошибки → снова веса → третья модель → и так далее.

Итог: каждая следующая модель всё сильнее фокусируется на трудных случаях.

### learning_rate (η)
Насколько сильно поднимать веса ошибочных точек.
- η = 1 → сильно, модель резко переобучается под ошибки
- η = 0.5 → слабее, обучение мягче

![Figure 6-8 — границы решений, learning rate 1 vs 0.5](6-8.png)

*Два НЕЗАВИСИМЫХ прогона по 5 моделей. Слева η = 1, справа η = 0.5.
Цифры 1, 2, 3, 5 — линии решений 1-й, 2-й, 3-й и 5-й модели внутри одного прогона.*

### Как предсказывает готовый ансамбль
Почти как bagging — голосование, но голоса НЕРАВНЫЕ:
модель поточнее голосует громче, слабая — тише.

### Сходство с градиентным спуском
GD подкручивает параметры одной модели. AdaBoost ничего не подкручивает —
он ДОБАВЛЯЕТ модели, и ансамбль становится лучше.

### ⚠ Минус (WARNING)
Распараллелить нельзя: модель №2 не начать, пока не обучена и не оценена №1.
Поэтому масштабируется хуже, чем bagging/pasting (там модели независимы).

### Про формулы подробнее

> «Каждый вес точки `w⁽ⁱ⁾` изначально равен `1/m`, так что их сумма равна 1.»

`m` — сколько точек в train. У нас 5 точек → у каждой вес `1/5 = 0.2`, сумма `1.0`.

```text
w⁽ⁱ⁾   вес точки i        y⁽ⁱ⁾  правильный ответ
ŷⱼ⁽ⁱ⁾  что сказала модель №j про точку i
rⱼ     ошибка модели №j   αⱼ    голос модели №j
η      learning rate      log   натуральный логарифм
exp(x) e^x                Σ     сумма
```

---

**1. Equation 6-1 — ошибка модели rⱼ**

> «Первая модель обучается, и на train считается её взвешенная ошибка r₁.»

```text
rⱼ =        Σ            w⁽ⁱ⁾
      i: ŷⱼ⁽ⁱ⁾ ≠ y⁽ⁱ⁾
```

```
Σ     — не по всем точкам, а только по тем, где условие под ней верно
w⁽ⁱ⁾  — вес точки (её «важность»)
≠     — сравнили предсказание с правдой
```

**Проще:** ошибка = сумма весов точек, где модель промахнулась. Точка весом 0.5 за промах «стоит» вдвое дороже точки весом 0.25.

**Пример.** Модель №1 ошиблась на точках 3 и 5:

```text
r₁ = 0.2 + 0.2 = 0.4
```

Диапазон `r`: 0 (идеал) … 1. `0.5` = уровень «монетка».

---

**2. Equation 6-2 — голос модели αⱼ**

> «Чем точнее модель, тем выше её вес. Если она просто угадывает — вес близок к нулю.
> Если чаще ошибается — вес отрицательный.»

```text
αⱼ = η · log( (1 − rⱼ) / rⱼ )
```

```
η        — learning rate, общий множитель (по умолчанию 1)
1 − rⱼ   — доля ПРАВИЛЬНЫХ ответов
rⱼ       — доля ошибок
дробь    — во сколько раз правильных больше, чем ошибок
log      — сжимает это отношение в удобное число
```

**Проще:** голос = «во сколько раз я лучше монетки», переведённое в число.

| `r` | правильных : ошибок | `α` | что значит |
|---|---|---|---|
| 0.10 | 9 : 1 | 2.20 | очень громкий голос |
| 0.40 | 1.5 : 1 | 0.405 | тихий голос |
| 0.50 | 1 : 1 | **0** | молчит |
| 0.75 | 1 : 3 | −1.10 | голосует наоборот |

**Пример.**

```text
α₁ = 1 · log( (1 − 0.4) / 0.4 ) = log(0.6 / 0.4) = log(1.5) = 0.405
```

`log(1.5) = 0.405` читается так: «`e` в степени `0.405` равно `1.5`». Логарифм и `exp` — обратные операции.

---

**3. Equation 6-3 — обновление весов точек**

> «AdaBoost поднимает веса неправильно классифицированных точек и заставляет следующую
> модель обращать на них больше внимания.»

```text
для i = 1, 2, ..., m:

        ┌ w⁽ⁱ⁾                    если ŷⱼ⁽ⁱ⁾ = y⁽ⁱ⁾   (угадала)
w⁽ⁱ⁾ ← ┤
        └ w⁽ⁱ⁾ · exp(αⱼ)          если ŷⱼ⁽ⁱ⁾ ≠ y⁽ⁱ⁾   (ошиблась)
```

Это **развилка**: правильные точки не трогаем, ошибочные умножаем.

**Почему именно `exp(α)`.** Формула 6-2 построена так, что `exp(α) = (1−r)/r`. У нас:

```text
exp(α₁) = exp(0.405) = 1.5    то же, что (1 − 0.4)/0.4 = 1.5
```

Множитель ровно равен «перевесу правильных над ошибками».

**Пример.**

```text
было:   [0.2, 0.2, 0.2, 0.2, 0.2]
точки 3 и 5 ошибочные:  0.2 × 1.5 = 0.3
стало:  [0.2, 0.2, 0.3, 0.2, 0.3]      сумма = 1.2
```

---

**4. Нормировка (сразу после 6-3)**

> «Затем все веса делятся на `Σ w⁽ⁱ⁾`, чтобы их сумма снова стала 1.»

```text
0.3 / 1.2 = 0.25      ← ошибочные стали тяжелее
0.2 / 1.2 = 0.167     ← правильные полегчали

итог: [0.167, 0.167, 0.25, 0.167, 0.25]   сумма = 1 ✔
```

Проверка: `3 · 0.167 + 2 · 0.25 = 0.5 + 0.5 = 1`.

**Зачем:** `r` — это сумма весов, и она имеет смысл только пока сумма всех весов = 1.

---

**5. Equation 6-4 — как ансамбль даёт ответ**

> «AdaBoost считает предсказания всех моделей и взвешивает их через αⱼ.
> Побеждает класс, набравший большинство взвешенных голосов.»

```text
ŷ(x) = argmax  Σ        αⱼ
           k   j: ŷⱼ(x)=k
```

```
argmax_k  — «какой номер класса k дал максимум»
Σ         — складываем α только тех моделей, которые назвали класс k
```

**Проще:** каждая модель кидает в урну своего класса не 1 голос, а `α` голосов.

**Пример.**

```text
модель 1: α = 0.4  → класс 1
модель 2: α = 1.1  → класс 1
модель 3: α = 0.9  → класс 0

класс 1: 0.4 + 1.1 = 1.5   ← максимум
класс 0: 0.9
ответ: класс 1
```

Не «две модели против одной», а **сумма голосов**.

---

**6. SAMME и код**

> «sklearn использует многоклассовую версию AdaBoost — SAMME. На двух классах она
> совпадает с AdaBoost. Базовая модель по умолчанию — decision stump (`max_depth=1`).»

```python
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1),   # пень: 1 вопрос, 2 листа
    n_estimators=30,
    learning_rate=0.5,
    random_state=42,
    algorithm="SAMME",
)
ada_clf.fit(X_train, y_train)
```

**TIP:** переобучается → уменьшить `n_estimators` или сильнее регуляризовать базовую модель.

---

**Вся цепочка одной картинкой**

```text
старт:  w = [0.2, 0.2, 0.2, 0.2, 0.2]
            ↓ модель 1 ошиблась на 3 и 5
6-1:    r₁ = 0.2 + 0.2 = 0.4
6-2:    α₁ = log(0.6/0.4) = 0.405        → голос модели 1
6-3:    w(3), w(5) = 0.2 · 1.5 = 0.3     → exp(α₁) = 1.5
норм:   [0.167, 0.167, 0.25, 0.167, 0.25]
            ↓ модель 2 учится на этих весах
6-4:    ŷ = класс с максимальной суммой α
```


## Gradient Boosting